# Phase 2: Training the ML Layer (EfficientNet, DistilBERT, RF, XGBoost, and SHAP)

This notebook trains and evaluates the core ML components of our system:
1. **Image Classifier**: Fine-tune **EfficientNet-B0** on HAM10000. Incorporate inverse class weighting to hit our **82-85%** accuracy target.
2. **NLP Text Classifiers**:
   - Fine-tune a lightweight **DistilBERT** on `Symptom2Disease` text (target **87-91%**).
   - Train a **Random Forest** on the 132-symptom structured binary dataset (target **85%+**).
3. **Severity Scorer**: Train an **XGBoost** model on patient vitals and demographics to calculate the risk score.
4. **SHAP Explainer**: Generate local and global explainability charts.

### Step 1: Train EfficientNet-B0 for Visual Skin Lesions

In [ ]:
import torch
import torch.nn as nn
from torchvision import models
import torch.optim as optim
from tqdm import tqdm
import numpy as np
from sklearn.metrics import classification_report, f1_score

# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 1. Define Model
class SkinLesionClassifier(nn.Module):
    def __init__(self, num_classes=7):
        super(SkinLesionClassifier, self).__init__()
        # Use weights parameter as pretrained is deprecated
        self.backbone = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        
        # Freeze backbone parameters initially to keep weights stable
        for param in self.backbone.features.parameters():
            param.requires_grad = True # Unfreeze for fine-tuning
            
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(p=0.4),
            nn.Linear(in_features, num_classes)
        )
        
    def forward(self, x):
        return self.backbone(x)

model = SkinLesionClassifier(num_classes=7).to(device)

# 2. Address Imbalance with Weighted Loss
# Calculate class weights: total_samples / (num_classes * class_samples)
# Using count estimates from HAM10000 classes:
class_counts = np.array([327, 514, 1099, 115, 6705, 142, 1113])
total_samples = sum(class_counts)
class_weights = total_samples / (7.0 * class_counts)
class_weights = torch.FloatTensor(class_weights).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

# 3. Training Loop
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in tqdm(loader, desc="Training"):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
    return running_loss / total, correct / total

def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    acc = np.mean(np.array(all_preds) == np.array(all_labels))
    f1 = f1_score(all_labels, all_preds, average='weighted')
    return running_loss / len(loader.dataset), acc, f1, all_labels, all_preds

# Executing Training for a sample 10 epochs
best_f1 = 0.0
for epoch in range(1, 11):
    # Assuming train_loader & val_loader are defined from Phase 1
    try:
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc, val_f1, _, _ = validate(model, val_loader, criterion, device)
        scheduler.step()
        
        print(f"Epoch {epoch}/10 | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} F1: {val_f1:.4f}")
        
        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), '/content/best_efficientnet_ham10000.pth')
            print("Saved new best model checkpoint!")
    except NameError:
        print("Dataloaders not defined. Please run Phase 1 notebook setup first.")
        break

### Step 2: Fine-tune DistilBERT on Symptom2Disease Text

In [ ]:
import pandas as pd
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from transformers import Trainer, TrainingArguments
from datasets import Dataset as HFDataset

# Load text data
s2d_df = pd.read_csv('/content/datasets/symptom2disease/Symptom2Disease.csv')
s2d_df = s2d_df.drop(columns=['Unnamed: 0'], errors='ignore')

# Map classes to numeric IDs
disease_labels = sorted(s2d_df['label'].unique())
disease2idx = {label: i for i, label in enumerate(disease_labels)}
idx2disease = {i: label for i, label in enumerate(disease_labels)}
s2d_df['label_idx'] = s2d_df['label'].map(disease2idx)

# Train/Test split
from sklearn.model_selection import train_test_split
train_texts, val_texts, train_labels, val_labels = train_test_split(
    s2d_df['text'].tolist(), s2d_df['label_idx'].tolist(), test_size=0.15, random_state=42, stratify=s2d_df['label_idx']
)

# Tokenize
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=128)

# HuggingFace Dataset Formats
class SymptomDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
        
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
        
    def __len__(self):
        return len(self.labels)

train_dataset = SymptomDataset(train_encodings, train_labels)
val_dataset = SymptomDataset(val_encodings, val_labels)

# Load Pre-trained Model
nlp_model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', 
    num_labels=len(disease_labels),
    id2label=idx2disease,
    label2id=disease2idx
)

# Configure Trainer
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1",
    fp16=True if torch.cuda.is_available() else False
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    f1 = f1_score(labels, preds, average='weighted')
    acc = np.mean(preds == labels)
    return {'eval_f1': f1, 'accuracy': acc}

trainer = Trainer(
    model=nlp_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

# Run training
trainer.train()
trainer.save_model('/content/saved_distilbert_symptom')

### Step 3: Train Random Forest on Structured Symptom Data

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import joblib

# Load 132-symptom structured binary dataset
df_train_binary = pd.read_csv('/content/datasets/symptom_binary/Training.csv')
df_test_binary = pd.read_csv('/content/datasets/symptom_binary/Testing.csv')

# Remove empty columns or clean prognoses
df_train_binary = df_train_binary.loc[:, ~df_train_binary.columns.str.contains('^Unnamed')]
df_test_binary = df_test_binary.loc[:, ~df_test_binary.columns.str.contains('^Unnamed')]

X_train_bin = df_train_binary.drop('prognosis', axis=1)
y_train_bin = df_train_binary['prognosis']
X_test_bin = df_test_binary.drop('prognosis', axis=1)
y_test_bin = df_test_binary['prognosis']

# Train Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=12)
rf_model.fit(X_train_bin, y_train_bin)

train_acc_rf = rf_model.score(X_train_bin, y_train_bin)
test_acc_rf = rf_model.score(X_test_bin, y_test_bin)
print(f"Random Forest — Train Acc: {train_acc_rf:.4f} | Test Acc: {test_acc_rf:.4f}")

# Save model
joblib.dump(rf_model, '/content/rf_symptom_binary.pkl')

### Step 4: Train XGBoost Severity Scorer

In [ ]:
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

severity_df = pd.read_csv('/content/datasets/severity/Disease_Diagnosis_and_Severity.csv')

# Encode Categorical values
le_gender = LabelEncoder()
severity_df['Gender'] = le_gender.fit_transform(severity_df['Gender'])

le_disease = LabelEncoder()
severity_df['Diagnosis'] = le_disease.fit_transform(severity_df['Diagnosis'])

le_severity = LabelEncoder()
severity_df['Severity_Encoded'] = le_severity.fit_transform(severity_df['Severity']) # Mild=0, Moderate=1, Severe=2

# Prepare features: Age, Gender, Temp, SpO2, HR, BP (Systolic), Diagnosis Code
# Adjust based on the actual columns present in your dataset
feature_cols = ['Age', 'Gender', 'Temperature', 'SpO2', 'Heart_Rate', 'Systolic_BP', 'Diagnosis']

# Check actual column names and rename if needed
print("Columns present:", severity_df.columns.tolist())

# Map column names if they differ
mapping = {
    'Heart Rate (bpm)': 'Heart_Rate',
    'SpO2 (%)': 'SpO2',
    'Temperature (°C)': 'Temperature',
    'Blood Pressure (systolic)': 'Systolic_BP'
}
severity_df = severity_df.rename(columns=mapping)

actual_features = [col for col in ['Age', 'Gender', 'Temperature', 'SpO2', 'Heart_Rate', 'Systolic_BP', 'Diagnosis'] if col in severity_df.columns]
print("Selected Features:", actual_features)

X_sev = severity_df[actual_features]
y_sev = severity_df['Severity_Encoded']

# Train/Val splits
X_train_sev, X_val_sev, y_train_sev, y_val_sev = train_test_split(
    X_sev, y_sev, test_size=0.2, random_state=42, stratify=y_sev
)

# Build XGBoost classifier
xgb_model = XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=6, random_state=42)
xgb_model.fit(X_train_sev, y_train_sev)

print(f"XGBoost Severity Classifier — Train Acc: {xgb_model.score(X_train_sev, y_train_sev):.4f} | Val Acc: {xgb_model.score(X_val_sev, y_val_sev):.4f}")
joblib.dump(xgb_model, '/content/xgb_severity_scorer.pkl')
joblib.dump(le_severity, '/content/label_encoder_severity.pkl')
joblib.dump(le_gender, '/content/label_encoder_gender.pkl')

### Step 5: SHAP Explainability Interpretation

In [ ]:
import shap

# Set up SHAP Explainer
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_val_sev)

# Plot global summary of feature impacts
shap.summary_plot(shap_values, X_val_sev)